In [2]:
from google.colab import files
uploaded = files.upload()

Saving AAPL.csv to AAPL.csv
Saving AMZN.csv to AMZN.csv
Saving GOOGL.csv to GOOGL.csv
Saving IBM.csv to IBM.csv
Saving JPM.csv to JPM.csv
Saving MSFT.csv to MSFT.csv
Saving NFLX.csv to NFLX.csv
Saving NVDA.csv to NVDA.csv
Saving ORCL.csv to ORCL.csv
Saving TSLA.csv to TSLA.csv


In [1]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.utils import class_weight

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
os.makedirs("models", exist_ok=True)
os.makedirs("scalers", exist_ok=True)
os.makedirs("metrics", exist_ok=True)

In [5]:
# ================================
# IMPORTS
# ================================
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.utils import class_weight

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, BatchNormalization, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# ================================
# CREATE FOLDERS
# ================================
os.makedirs("models", exist_ok=True)
os.makedirs("scalers", exist_ok=True)
os.makedirs("metrics", exist_ok=True)

results = []

# ================================
# TRAIN LOOP
# ================================
for file in uploaded.keys():
    print(f"\nProcessing {file}...")

    stock = file.replace(".csv", "")
    df = pd.read_csv(file)

    # -------------------------
    # BASIC
    # -------------------------
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date')

    # -------------------------
    # 🔥 TARGET (LESS NOISE)
    # -------------------------
    df['Target'] = (df['Close'].shift(-1) > df['Close'] * 1.001).astype(int)

    # -------------------------
    # 🔥 FEATURES
    # -------------------------
    df['Return'] = df['Close'].pct_change(fill_method=None)*200
    df['MA5'] = df['Close'].rolling(5).mean()
    df['MA10'] = df['Close'].rolling(10).mean()
    df['Diff'] = (df['Close'] - df['Open']) * 10
    df['Range'] = (df['High'] - df['Low']) * 10

    # 🔥 NEW FEATURE (VERY IMPORTANT)
    df['Trend'] = (df['MA5'] - df['MA10']) * 10

    # -------------------------
    # CLEAN DATA
    # -------------------------
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna()

    features = [
        'Open', 'High', 'Low', 'Close', 'Volume',
        'Return', 'MA5', 'MA10', 'Diff', 'Range',
        'Trend'
    ]

    X = df[features]
    y = df['Target']

    # -------------------------
    # TRAIN-TEST SPLIT
    # -------------------------
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, shuffle=True
    )

    # -------------------------
    # SCALING
    # -------------------------
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # -------------------------
    # 🔥 CLASS BALANCING
    # -------------------------
    class_weights = class_weight.compute_class_weight(
        class_weight='balanced',
        classes=np.unique(y_train),
        y=y_train
    )
    class_weights = dict(enumerate(class_weights))

    # -------------------------
    # 🔥 ANN MODEL
    # -------------------------
    model = Sequential([
        Input(shape=(X_train.shape[1],)),

        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.2),

        Dense(32, activation='relu'),
        BatchNormalization(),

        Dense(16, activation='relu'),

        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    # -------------------------
    # EARLY STOPPING
    # -------------------------
    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    )

    # -------------------------
    # TRAIN
    # -------------------------
    model.fit(
        X_train, y_train,
        epochs=80,
        batch_size=32,
        validation_split=0.2,
        callbacks=[early_stop],
        class_weight=class_weights,
        verbose=0
    )

    # -------------------------
    # 🔥 THRESHOLD TUNING
    # -------------------------
    y_prob = model.predict(X_test)

    best_acc = 0
    best_thresh = 0.5

    for t in np.arange(0.45, 0.75, 0.01):
        y_pred_temp = (y_prob > t).astype(int)
        acc_temp = accuracy_score(y_test, y_pred_temp)

        if acc_temp > best_acc:
            best_acc = acc_temp
            best_thresh = t

    # FINAL PREDICTION
    y_pred = (y_prob > best_thresh).astype(int)

    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)

    print(f"{stock} Accuracy:", acc)
    print(f"Best Threshold: {best_thresh}")
    print("Confusion Matrix:\n", cm)

    # -------------------------
    # SAVE
    # -------------------------
    model.save(f"models/{stock}_model.keras")
    joblib.dump(scaler, f"scalers/{stock}_scaler.pkl")
    np.save(f"metrics/{stock}_cm.npy", cm)

    results.append((stock, acc))

# ================================
# SAVE RESULTS
# ================================
results_df = pd.DataFrame(results, columns=["Stock", "Accuracy"])
results_df.to_csv("metrics/Accuracy.csv", index=False)

print("\nFinal Results:")
print(results_df)


Processing AAPL.csv...
62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
AAPL Accuracy: 0.5348484848484848
Best Threshold: 0.55
Confusion Matrix:
 [[1024   30]
 [ 891   35]]

Processing AMZN.csv...
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step
AMZN Accuracy: 0.5104347826086957
Best Threshold: 0.6600000000000001
Confusion Matrix:
 [[585   2]
 [561   2]]

Processing GOOGL.csv...
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step
GOOGL Accuracy: 0.5452229299363057
Best Threshold: 0.51
Confusion Matrix:
 [[242 148]
 [209 186]]

Processing IBM.csv...
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
IBM Accuracy: 0.526100307062436
Best Threshold: 0.5700000000000001
Confusion Matrix:
 [[1457   66]
 [1323   85]]

Processing JPM.csv...
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
JPM Accuracy: 0.5252725470763132
Best Threshold: 0.6500000000000001
Confusion Matrix:
 [[1055    3]
 [ 955    5]]

Processing MSFT.csv...
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step
MSFT Accuracy: 0.5422740524781341
Best Threshold: 0.51
Confusion Matrix:
 [[729 180]


In [ ]:
import shutil

shutil.make_archive("models", 'zip', "models")
shutil.make_archive("scalers", 'zip', "scalers")
shutil.make_archive("metrics", 'zip', "metrics")

files.download("models.zip")
files.download("scalers.zip")
files.download("metrics.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>